In [7]:
import sys
sys.path.append('../')

import torch
import json

from src.model import Transformer, generate_batch
from src.dataset import (
    NMRDataset, 
    load_multiplicity_codes, 
    load_split, 
    collate_fn
)

import sentencepiece as spm

from functools import partial

/home/nick1899/miniconda3/envs/nmr_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
tokenizer = spm.SentencePieceProcessor('../checkpoints/nmr2struct.model')

In [9]:
model = Transformer(
    multiplicity_vocab_size=len(NMRDataset.MULTIPLICITY2IDX),
    multiplicity_hidden_dim=48,
    spectrum_d_ff=256,
    spectrum_hidden_dim=104,
    intensity_d_ff=256,
    intensity_hidden_dim=104,
    tgt_vocab_size=tokenizer.vocab_size(),
    d_model=256,
    num_heads=8,
    num_layers=6,
    d_ff=512,
    max_seq_length=400,
    dropout=0.2,
    smiles_pad_token=tokenizer.pad_id()
    )
model.load_state_dict(torch.load('../checkpoints/bimodal_256_new_data_full_155.pt', map_location='cpu'))
model.eval()

Transformer(
  (spectrum_embedding): SpectrumEmbedding(
    (spectrum_embs): ModuleDict(
      (C_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
      (H_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
    )
    (intensity_embs): ModuleDict(
      (C_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
      (H_NMR): FourierEmbedding(
        (proj): Linear(in_features=256, out_features=104, bias=True)
      )
    )
    (multiplicity_embs): ModuleDict(
      (C_NMR): Embedding(10, 48)
      (H_NMR): Embedding(10, 48)
    )
  )
  (decoder_embedding): Embedding(512, 256)
  (positional_encoding): PositionalEncoding()
  (encoder_carbon_layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (W_q): Linear(in_features=256, out_features=256, bias=True)
        (W_k): Linear(in_features=256, out_featu

In [10]:
multiplicity_codes = load_multiplicity_codes('../data/multiplicity_codes.json')

In [11]:
test_data = load_split(
    path='../data/test_full_50k.jsonl', 
    multiplicity_codes=multiplicity_codes,
    solvent='CDCl3' # Так собраны данные
)

Loading test_full_50k.jsonl: 50000it [00:01, 27562.39it/s]
Processing spectra:  36%|███▋      | 3619/9954 [00:00<00:01, 3778.04it/s]

Processing spectra: 100%|██████████| 9954/9954 [00:02<00:00, 3581.33it/s]


In [12]:
# Для каждой молекулы есть разное количество 1H и 13C спектров
# Датасет устроен так что он каждый раз выдает случайную пару спектров
# И с вероятностью 20% маскирует один из спектров
# Если хочется работать иначе с этим - см. src/dataset.py L153

test_dataset = NMRDataset(
    data=test_data,
    solvent='CDCl3',
    tokenizer=tokenizer,
    smiles_bos_id=tokenizer.bos_id(),
    smiles_eos_id=tokenizer.eos_id(),
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, 
    shuffle=False, 
    batch_size=4, 
    collate_fn=partial(
        collate_fn, 
        smiles_pad_id=tokenizer.pad_id(),
        spectrum_pad_token=-1000.,
        max_smiles_len=400
    )
)

In [13]:
data = next(iter(test_loader))

smiles_predictions, saved_hidden_states = generate_batch(
    model=model,
    src_c_nmr=data['C_NMR'],
    src_h_nmr=data['H_NMR'],
    tokenizer=tokenizer,
    max_tokens=400,
    device=torch.device("cpu"), 
    return_decoder_states=True
)

In [14]:
decoder_states = saved_hidden_states
# decoder_states: List[Tensor], len = 6, each [B, T, D] = [4, 17, 256]
B = decoder_states[0].size(0)   # 4
T = decoder_states[0].size(1)   # 17
D = decoder_states[0].size(2)   # 256
# stack to [L, B, T, D] = [6, 4, 17, 256]
stacked = torch.stack(decoder_states, dim=0)
# build per-sample list: len = B, each [L, T, D] = [6, 17, 256]
per_sample_states = [stacked[:, b, :, :] for b in range(B)]
# per_sample_states[b].shape -> torch.Size([6, 17, 256])

In [15]:
true_smiles_list = tokenizer.decode(data['smiles'].squeeze().numpy().tolist())

In [16]:
for pred_smiles, true_smiles in zip(smiles_predictions, tokenizer.decode(data['smiles'].squeeze().numpy().tolist())):
    print(f"True SMILES: {true_smiles}", f"Predicted SMILES: {pred_smiles}", sep='\t')

True SMILES: O=C1OC(=O)c2ccccc21	Predicted SMILES: O=C1c2ccccc2C(=O)N1Cl
True SMILES: Clc1nc(Cl)c2ccccc2n1	Predicted SMILES: O=C(Cl)c1nc2ccccc2s1
True SMILES: c1ccc2c(N3CCCC3)ncnc2c1	Predicted SMILES: c1ccc2c(N3CCCC3)ncnc2c1
True SMILES: COc1ccc(C=O)cc1OC	Predicted SMILES: COc1ccc(C=O)cc1OC


In [17]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, Fragments

FRAGMENT_FUNCS = [
    getattr(Fragments, name)
    for name in dir(Fragments)
    if name.startswith("fr_") and callable(getattr(Fragments, name))
]

def make_bool(val):
    if val != 0:
        return 1
    else:
        return 0 

def calculate_target(smiles_list: list[str]) -> list[list[float]]:
    result = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            result.append([0.0] * (8 + len(FRAGMENT_FUNCS)))
            continue

        ring_feats = [
            rdMolDescriptors.CalcNumRings(mol),
            rdMolDescriptors.CalcNumAliphaticRings(mol),
            rdMolDescriptors.CalcNumAliphaticHeterocycles(mol),
            rdMolDescriptors.CalcNumAliphaticCarbocycles(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
            rdMolDescriptors.CalcNumAromaticHeterocycles(mol),
            rdMolDescriptors.CalcNumAromaticCarbocycles(mol),
            rdMolDescriptors.CalcNumHeterocycles(mol),
        ]

        frag_feats = [func(mol) for func in FRAGMENT_FUNCS]
        res_for_one_smi = ring_feats + frag_feats

        res_for_one_smi = [make_bool(i) for i in res_for_one_smi]
        result.append(res_for_one_smi)
    return result

In [18]:
from torch.utils.data import DataLoader
import torch
from typing import List

# assumes `calculate_target(smiles: str) -> list[float]` is defined elsewhere in the notebook

class HiddenStatesProbingDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        hidden_states: List[torch.Tensor] | torch.Tensor,
        smiles_list: List[str],
        hidden_state_idx: int = 0,
        descriptor_idx: int = 0,
    ) -> None:
        """
        hidden_states: list of [d_model] tensors
        smiles_list: list of SMILES strings, same length and order as hidden_states
        """
        hidden_states = [torch.mean(i[hidden_state_idx,:,:],0) for i in hidden_states] # берём хидден стейт с определённого уровня аттеншена

        
        if isinstance(hidden_states, list):
            self.x = torch.stack(hidden_states, dim=0)  # [N, d_model]
        else:
            self.x = hidden_states  # [N, d_model]

        self.smiles_list = smiles_list
        self.descriptor_list = [(i[descriptor_idx]) for i in calculate_target(smiles_list)]

    def __len__(self) -> int:
        return self.x.size(0)

    def __getitem__(self, idx: int):
        x = self.x[idx]  # [d_model]

        smiles = self.smiles_list[idx] # for debug simplicity

        desc_values = self.descriptor_list[idx]  
        y = torch.tensor(desc_values, dtype=torch.float32)

        return {"x": x, "y": y, "smiles": smiles}

In [19]:
test_probbing_dataset = HiddenStatesProbingDataset(
    hidden_states=per_sample_states,
    smiles_list=true_smiles_list,
    descriptor_idx=5
)

test_probbing_loader = torch.utils.data.DataLoader(
    test_probbing_dataset, 
    shuffle=False, 
    batch_size=4
)

In [20]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F


class LinearProbe(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.fc = nn.Linear(input_dim, 2)        # 2 classes: 0 and 1
        self.softmax = nn.Softmax(dim=-1)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.fc(x)                      # [B, 2]
        probs = self.softmax(logits)             # [B, 2], sums to 1
        return probs

def train_linear_probe(
    loader: torch.utils.data.DataLoader,
    batch_size: int = 4,
    lr: float = 1e-3,
    epochs: int = 20,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):

    model = LinearProbe(input_dim=256).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        n = 0

        for batch in loader:
            x = batch["x"].to(device)          # [B, d_model]
            y = batch["y"].to(device).long()          # [B]

            optimizer.zero_grad()
            probs = model(x)                   # [B]
            y_onehot = F.one_hot(y, num_classes=2).float()  # [B, 2]
            loss = criterion(probs, y_onehot)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            n += x.size(0)

        print(f"Epoch {epoch:02d} | train_loss = {total_loss / n:.4f}")

    return model

In [21]:

probe_model = train_linear_probe(test_probbing_dataset, batch_size=4, lr=1e-3, epochs=20)

Epoch 01 | train_loss = 0.7080
Epoch 02 | train_loss = 0.6909
Epoch 03 | train_loss = 0.6795
Epoch 04 | train_loss = 0.6687
Epoch 05 | train_loss = 0.6582
Epoch 06 | train_loss = 0.6480
Epoch 07 | train_loss = 0.6381
Epoch 08 | train_loss = 0.6284
Epoch 09 | train_loss = 0.6189
Epoch 10 | train_loss = 0.6095
Epoch 11 | train_loss = 0.6004
Epoch 12 | train_loss = 0.5915
Epoch 13 | train_loss = 0.5828
Epoch 14 | train_loss = 0.5743
Epoch 15 | train_loss = 0.5660
Epoch 16 | train_loss = 0.5578
Epoch 17 | train_loss = 0.5498
Epoch 18 | train_loss = 0.5419
Epoch 19 | train_loss = 0.5342
Epoch 20 | train_loss = 0.5267


In [62]:
test_probbing_dataset[2]['y']

tensor(1.)